In [5]:
import warnings
from statsmodels.tsa.stattools import adfuller
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss
import sklearn.model_selection as ms
import yfinance as yf

# Tắt cảnh báo có chọn lọc
warnings.filterwarnings("ignore", category=UserWarning)

print("==================================================")
print(" KHỞI CHẠY HỆ THỐNG ĐỊNH LƯỢNG KRW/VND (V4.1 - FIXED)")
print("==================================================")

# =====================================================================
# TẦNG 1: DATA & CLEANING LAYER (Xử lý Stale Data & Lọc Outlier chuẩn)
# =====================================================================
print(
    "\n--- [Tầng 1] Tải dữ liệu 5 năm, lọc Stale Data và cấu hình chi phí ---"
)

usd_krw = yf.download("KRW=X", period="5y", interval="1d", auto_adjust=True)[
    "Close"
]
usd_vnd = yf.download("VND=X", period="5y", interval="1d", auto_adjust=True)[
    "Close"
]

if isinstance(usd_krw, pd.DataFrame):
  usd_krw = usd_krw.iloc[:, 0]
if isinstance(usd_vnd, pd.DataFrame):
  usd_vnd = usd_vnd.iloc[:, 0]

df = pd.DataFrame({"USD_KRW": usd_krw, "USD_VND": usd_vnd}).dropna()
raw_prices = df["USD_VND"] / df["USD_KRW"]

if raw_prices.index.tz is not None:
  raw_prices.index = raw_prices.index.tz_localize(None)
raw_prices.index = pd.to_datetime(raw_prices.index).normalize()

# Loại bỏ Stale Data (ngày giá không đổi, diff == 0) và spike bất thường (> 1.5%)
raw_prices = raw_prices[raw_prices.diff() != 0].dropna()
raw_prices = raw_prices[raw_prices.pct_change().abs() < 0.015].dropna()

log_prices = np.log(raw_prices)
ROUND_TRIP_COST = 0.0040  # 0.4% chi phí khứ hồi thực tế


# =====================================================================
# TẦNG 2: OPTIMIZED FFD LAYER (Trên Log-Prices)
# =====================================================================
print("\n--- [Tầng 2] Quét tìm bậc d tối ưu trên Log-Prices ---")


def get_weights_ffd(d, thres=1e-3):
  w = [1.0]
  k = 1
  while True:
    w_ = -w[-1] / k * (d - k + 1)
    if abs(w_) < thres:
      break
    w.append(w_)
    k += 1
  return np.array(w[::-1])


def fractional_diff_ffd(series, d, thres=1e-3):
  w = get_weights_ffd(d, thres)
  width = len(w) - 1
  df_res = pd.Series(index=series.index, dtype=float)
  for iloc in range(width, len(series)):
    loc = series.index[iloc]
    sub_series = series.iloc[iloc - width : iloc + 1]
    df_res.loc[loc] = np.dot(w, sub_series.values)
  return df_res.dropna()


best_d = 0.05
tested_ds = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
for d_test in tested_ds:
  test_series = fractional_diff_ffd(log_prices, d=d_test)
  if len(test_series) > 100:
    adf_result = adfuller(test_series)
    if adf_result[1] < 0.05:
      best_d = d_test
      break
    best_d = d_test

print(f" -> Bậc d tối ưu được chọn: {best_d}")
stationary_series = fractional_diff_ffd(log_prices, d=best_d)


# =====================================================================
# TẦNG 3: TRUE META-MODEL LAYER (Mở rộng Barrier & Holding Period)
# =====================================================================
print(
    "\n--- [Tầng 3] Xây dựng đặc trưng toàn cục (Dùng .diff() cho Return) ---"
)

clean_features = pd.DataFrame(index=log_prices.index, dtype=float)
clean_features["Log_Price"] = log_prices
clean_features["Frac_Diff"] = stationary_series
clean_features["Rolling_Mean"] = stationary_series.rolling(window=5).mean()
clean_features["Rolling_Std"] = stationary_series.rolling(window=5).std()
clean_features["Return_1d"] = log_prices.diff()

# Primary Model: Momentum 10 phiên
primary_signal = np.where(log_prices.diff(10) > 0, 1, -1)
clean_features["Primary_Side"] = pd.Series(primary_signal, index=log_prices.index)


def get_daily_vol(close, span=30):
  return close.pct_change().ewm(span=span).std().dropna()


daily_vol = get_daily_vol(raw_prices)


def apply_triple_barrier(prices_series, events, pt_sl):
  out = pd.DataFrame(index=events.index)
  out["ret"] = np.nan
  out["bin"] = 0
  for loc, t1 in events["t1"].items():
    path = prices_series.loc[loc:t1]
    if len(path) <= 1:
      continue
    base_val = prices_series.loc[loc]
    ret = (path / base_val) - 1
    side = events.loc[loc, "side"]
    ret_directional = ret * side
    target = max(events.loc[loc, "trgt"], 0.001)

    pt = target * pt_sl[0]
    sl = -target * pt_sl[1]

    hit_pt = ret_directional[ret_directional >= pt].index.min()
    hit_sl = ret_directional[ret_directional <= sl].index.min()

    if pd.notna(hit_pt) and (pd.isna(hit_sl) or hit_pt < hit_sl):
      out.loc[loc, "bin"] = 1
      out.loc[loc, "ret"] = ret.loc[hit_pt] * side
    elif pd.notna(hit_sl) and (pd.isna(hit_pt) or hit_sl < hit_pt):
      out.loc[loc, "bin"] = -1
      out.loc[loc, "ret"] = ret.loc[hit_sl] * side
    else:
      out.loc[loc, "bin"] = 0
      out.loc[loc, "ret"] = ret.iloc[-1] * side
  return out.dropna(subset=["ret"])


holding_period = 15
dropped_features = clean_features.dropna()
events_index = dropped_features.index[30:-holding_period]
events = pd.DataFrame(index=events_index)
events["trgt"] = daily_vol.loc[events.index]
events["side"] = dropped_features.loc[events.index, "Primary_Side"]
events["t1"] = raw_prices.index[
    [
        min(
            raw_prices.index.get_loc(idx) + holding_period,
            len(raw_prices) - 1,
        )
        for idx in events.index
    ]
]

labels = apply_triple_barrier(raw_prices, events, pt_sl=[2.0, 2.0])

valid_idx = labels.index.intersection(dropped_features.index)
X = dropped_features.loc[
    valid_idx,
    ["Frac_Diff", "Rolling_Mean", "Rolling_Std", "Return_1d", "Primary_Side"],
]
meta_y = (labels.loc[valid_idx, "ret"] > 0).astype(int)

print(f" -> Số lượng mẫu dữ liệu sạch sau xử lý: {len(X)}")
print(
    f" -> Phân phối nhãn Meta-y (1: Thắng, 0: Thua/Hòa):"
    f" {meta_y.value_counts().to_dict()}"
)


# =====================================================================
# TẦNG 4: RIGOROUS VALIDATION LAYER (Đánh giá Purged CV sạch lỗi Index)
# =====================================================================
print("\n--- [Tầng 4] Kiểm định chéo Purged CV & Hiệu chuẩn xác suất ---")


class PurgedKFold(ms.KFold):

  def __init__(self, n_splits=5, t1=None, embargo_pct=0.01):
    super(PurgedKFold, self).__init__(
        n_splits=n_splits, shuffle=False, random_state=None
    )
    self.t1 = t1
    self.embargo_pct = embargo_pct

  def split(self, X, y=None, groups=None):
    if self.t1 is None:
      raise ValueError("Cần chuỗi t1")
    for train_index, test_index in super().split(X, y):
      test_start = X.index[test_index[0]]
      test_end = X.index[test_index[-1]]
      train_times = self.t1.loc[X.index[train_index]]

      banned_indices = train_times[
          (train_times.index <= test_end) & (train_times >= test_start)
      ].index
      train_index = np.array(
          [i for i in train_index if X.index[i] not in banned_indices]
      )

      embargo_idx = int(X.shape[0] * self.embargo_pct)
      if embargo_idx > 0:
        test_end_loc = X.index.get_loc(test_end)
        embargo_end_loc = min(test_end_loc + embargo_idx, X.shape[0])
        embargo_dates = X.index[test_end_loc + 1 : embargo_end_loc]
        train_index = np.array(
            [i for i in train_index if X.index[i] not in embargo_dates]
        )
      yield train_index, test_index


t1_series = events["t1"].loc[X.index]
cv = PurgedKFold(n_splits=5, t1=t1_series, embargo_pct=0.01)

brier_scores = []
baseline_briers = []

for train_idx, test_idx in cv.split(X):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = meta_y.iloc[train_idx], meta_y.iloc[test_idx]

  fold_model = RandomForestClassifier(
      n_estimators=100, max_depth=3, random_state=42, class_weight="balanced"
  )
  fold_model.fit(X_train, y_train)
  probs = fold_model.predict_proba(X_test)[:, 1]

  brier_scores.append(brier_score_loss(y_test, probs))
  p_mean = y_train.mean()
  baseline_briers.append(
      brier_score_loss(y_test, np.full_like(y_test, p_mean, dtype=float))
  )

print(
    f" -> Brier Score qua CV: {np.mean(brier_scores):.4f} (±"
    f" {np.std(brier_scores):.4f})"
)
print(
    f" -> Brier Baseline (p): {np.mean(baseline_briers):.4f} (±"
    f" {np.std(baseline_briers):.4f})"
)

# Khởi tạo mô hình hiệu chuẩn toàn cục bằng cách truyền trực tiếp splitter `cv` vào CalibratedClassifierCV
base_model = RandomForestClassifier(
    n_estimators=100, max_depth=3, random_state=42, class_weight="balanced"
)
meta_model = CalibratedClassifierCV(base_model, method="sigmoid", cv=cv)
meta_model.fit(X, meta_y)


# =====================================================================
# TẦNG 5: EXPECTED VALUE (EV) & DIAGNOSTIC METRICS
# =====================================================================
print("\n--- [Tầng 5] Phân tích EV, Ngưỡng hòa vốn p* và Tín hiệu Live ---")

clean_features_fixed = clean_features.dropna()
current_data = clean_features_fixed.iloc[[-1]]
latest_date = current_data.index[0].strftime("%Y-%m-%d")
current_price = raw_prices.loc[current_data.index[0]]

pred_prob = meta_model.predict_proba(
    current_data[
        [
            "Frac_Diff",
            "Rolling_Mean",
            "Rolling_Std",
            "Return_1d",
            "Primary_Side",
        ]
    ]
)[:, 1][0]
current_side = current_data["Primary_Side"].values[0]

historical_gains = labels.loc[labels["ret"] > 0, "ret"].abs().mean()
historical_losses = labels.loc[labels["ret"] < 0, "ret"].abs().mean()
if pd.isna(historical_gains) or historical_gains == 0:
  historical_gains = 0.01
if pd.isna(historical_losses) or historical_losses == 0:
  historical_losses = 0.01

p_breakeven = (historical_losses + ROUND_TRIP_COST) / (
    historical_gains + historical_losses
)

p_win = pred_prob
p_loss = 1 - pred_prob
expected_value = (
    (p_win * historical_gains)
    - (p_loss * historical_losses)
    - ROUND_TRIP_COST
)

print("\n" + "=" * 65)
print(f" BẢN TIN CHẨN ĐOÁN ĐỊNH LƯỢNG - NGÀY: {latest_date}")
print("=" * 65)
print(f" Tỷ giá KRW/VND tham chiếu: {current_price:.4f}")
print(f" Lợi nhuận trung bình (Gain): {historical_gains * 100:.2f}%")
print(f" Thua lỗ trung bình (Loss): {historical_losses * 100:.2f}%")
print(f" Chi phí khứ hồi (Cost): {ROUND_TRIP_COST * 100:.2f}%")
print(
    f" Ngưỡng xác suất hòa vốn (p*): {p_breakeven * 100:.2f}% (Model cần"
    f" p > p*)"
)
print("-" * 65)
print(f" Xác suất thành công của Model (p): {pred_prob * 100:.2f}%")
print(f" Kỳ vọng lợi nhuận thuần (EV): {expected_value * 100:.4f}%")

if expected_value > 0 and pred_prob > p_breakeven:
  print(" 🟢 TÍN HIỆU: ĐÈN XANH (EV > 0, Đã vượt điểm hòa vốn p*)")
else:
  print(
      " 🔴 TÍN HIỆU: ĐÈN ĐỎ (EV <= 0 hoặc p <= p* / Đứng ngoài bảo toàn vốn)"
  )
print("=" * 65)
print(" * Lưu ý thực thi: Tín hiệu tính từ Close ngày t, khớp lệnh tại t+1.")

 KHỞI CHẠY HỆ THỐNG ĐỊNH LƯỢNG KRW/VND (V4.1 - FIXED)

--- [Tầng 1] Tải dữ liệu 5 năm, lọc Stale Data và cấu hình chi phí ---


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



--- [Tầng 2] Quét tìm bậc d tối ưu trên Log-Prices ---
 -> Bậc d tối ưu được chọn: 0.05

--- [Tầng 3] Xây dựng đặc trưng toàn cục (Dùng .diff() cho Return) ---
 -> Số lượng mẫu dữ liệu sạch sau xử lý: 1168
 -> Phân phối nhãn Meta-y (1: Thắng, 0: Thua/Hòa): {1: 592, 0: 576}

--- [Tầng 4] Kiểm định chéo Purged CV & Hiệu chuẩn xác suất ---
 -> Brier Score qua CV: 0.2475 (± 0.0033)
 -> Brier Baseline (p): 0.2506 (± 0.0010)

--- [Tầng 5] Phân tích EV, Ngưỡng hòa vốn p* và Tín hiệu Live ---

 BẢN TIN CHẨN ĐOÁN ĐỊNH LƯỢNG - NGÀY: 2026-11-03
 Tỷ giá KRW/VND tham chiếu: 19.1755
 Lợi nhuận trung bình (Gain): 1.49%
 Thua lỗ trung bình (Loss): 1.43%
 Chi phí khứ hồi (Cost): 0.40%
 Ngưỡng xác suất hòa vốn (p*): 62.59% (Model cần p > p*)
-----------------------------------------------------------------
 Xác suất thành công của Model (p): 43.31%
 Kỳ vọng lợi nhuận thuần (EV): -0.5628%
 🔴 TÍN HIỆU: ĐÈN ĐỎ (EV <= 0 hoặc p <= p* / Đứng ngoài bảo toàn vốn)
 * Lưu ý thực thi: Tín hiệu tính từ Close ngày 